In [1]:
# ────────────────────────────────────────────────────────────
# Imports & config
# ────────────────────────────────────────────────────────────
import pyarrow.parquet as pq
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from pathlib import Path
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import IterableDataset, DataLoader
from tqdm.auto import tqdm
from multiprocessing import Manager
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    roc_auc_score, classification_report
)
import sys
import torch.multiprocessing
import multiprocessing as mp
import threading
import time
from multiprocessing import Queue
torch.multiprocessing.set_start_method('spawn', force=True)

C:\Users\taren\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
# ---- paths & basic hyper‐params ----------------------------------------------
parquet_path = Path("MasterDS/Master_cleaned.parquet")
return_col   = "ret_5d_future"       # target column
batch_size   = 64
n_epochs     = 15
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using {device}")

# ────────────────────────────────────────────────────────────
# 0) Load parquet metadata + train/test split
# ────────────────────────────────────────────────────────────
pf = pq.ParquetFile(parquet_path)

# 0-A  collect ALL unique dates (streaming)
all_dates = []
for rg in range(pf.num_row_groups):
    col = pf.read_row_group(rg, columns=["date"]).to_pandas()["date"]
    all_dates.extend(col.unique())

all_dates = sorted(set(all_dates))
n_total   = len(all_dates)
train_idx = int(n_total * 0.80)           # 80 % train, 20 % test

train_dates = set(all_dates[:train_idx])
test_dates  = set(all_dates[train_idx:])

print(f"Dates→ train {len(train_dates)}  |  test {len(test_dates)}")

# ────────────────────────────────────────────────────────────
# 1) Feature columns & ticker mapping
# ────────────────────────────────────────────────────────────
all_cols     = pf.schema_arrow.names
exclude_cols = {return_col, "date", "ticker"}
feature_cols = [c for c in all_cols if c not in exclude_cols]
print(f"{len(feature_cols)} numeric feature columns")

# ticker vocab
unique_tickers = set()
for rg in range(pf.num_row_groups):
    unique_tickers.update(
        pf.read_row_group(rg, columns=["ticker"])
          .to_pandas()["ticker"].unique()
    )
ticker_id_map = {t: i for i, t in enumerate(sorted(unique_tickers))}
ticker_vocab  = len(ticker_id_map)
print(f"{ticker_vocab} unique tickers")

# ────────────────────────────────────────────────────────────
# 2) Fit StandardScaler **on train rows only**
# ────────────────────────────────────────────────────────────
sample_frames = []
for i in range(min(20, pf.num_row_groups)):      # sample ≤ 20 row-groups
    df_rg = pf.read_row_group(i).to_pandas()
    df_rg = df_rg[df_rg["date"].isin(train_dates)]
    if df_rg.empty:
        continue
    sample_frames.append(
        df_rg.sample(n=3_000, random_state=42)    # 3 k rows each
    )
calib_df = pd.concat(sample_frames, axis=0)
feature_scaler = StandardScaler().fit(calib_df[feature_cols].values)

del sample_frames, calib_df

# ────────────────────────────────────────────────────────────
# 3) Build 95-th percentile cutoff map **from train only**
# ────────────────────────────────────────────────────────────
values_by_date = defaultdict(list)

for batch in pf.iter_batches(batch_size=1_000_000):
    dfb = batch.to_pandas()[["date", return_col]]
    for d, grp in dfb.groupby("date"):
        values_by_date[d].extend(grp[return_col].values.tolist())

cutoff_map = {d: np.quantile(v, 0.95) for d, v in values_by_date.items()}
print(f"Cutoff map for {len(cutoff_map)} dates (train + test)")
del values_by_date


# ────────────────────────────────────────────────────────────
# 5) Optimized Dataset with better multiprocessing support
# ────────────────────────────────────────────────────────────
class ParquetSequenceDataset(IterableDataset):
    """
    Optimized version with better batching and memory management
    """
    def __init__(self, date_subset, feature_cols, return_col,
                 cutoff_map, feature_scaler,
                 window=10, pos_ratio=None, batch_size=4096):
        super().__init__()
        self.dates          = set(date_subset)
        self.feature_cols   = feature_cols
        self.return_col     = return_col
        self.cutoff_map     = cutoff_map
        self.feature_scaler = feature_scaler
        self.window         = window
        self.pos_ratio      = pos_ratio
        self.batch_size     = batch_size
        
        # Pre-compute worker info
        self.worker_id = None
        self.num_workers = None

    def __iter__(self):
        # Get worker info
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            self.worker_id = worker_info.id
            self.num_workers = worker_info.num_workers
        else:
            self.worker_id = 0
            self.num_workers = 1
            
        pf_local = pq.ParquetFile("MasterDS/Master_cleaned.parquet")
        history  = defaultdict(lambda: deque(maxlen=self.window))
        
        # Use larger internal batch size for efficiency
        internal_batch_size = self.batch_size * 8
        buf_X, buf_y, buf_meta = [], [], []
        
        # Process row groups in round-robin fashion for workers
        for rg_idx in range(self.worker_id, pf_local.num_row_groups, self.num_workers):
            try:
                # Read entire row group at once
                df = pf_local.read_row_group(rg_idx).to_pandas()
                df = df[df["date"].isin(self.dates)]
                if df.empty:
                    continue
                
                # Process in chunks to manage memory
                chunk_size = 50000
                for chunk_start in range(0, len(df), chunk_size):
                    chunk = df.iloc[chunk_start:chunk_start + chunk_size]
                    
                    # Ensure chronological windows per ticker
                    chunk = chunk.sort_values(["ticker", "date"])
                    
                    # Label using precomputed cutoff_map
                    chunk["label"] = (chunk[self.return_col] >= chunk["date"].map(self.cutoff_map)).astype(int)
                    
                    # Process chunk
                    for _, row in chunk.iterrows():
                        key = row["ticker"]
                        feats = row[self.feature_cols].to_numpy(dtype=np.float32)
                        
                        # Append current features to history
                        hist = history[key]
                        hist.append(feats)
                        if len(hist) < self.window:
                            continue
                        
                        # Build sample
                        X_seq = np.stack(hist, axis=0)
                        buf_X.append(X_seq)
                        buf_y.append(float(row["label"]))
                        buf_meta.append((row["date"], key))
                        
                        # Flush when we have enough samples
                        if len(buf_y) >= internal_batch_size:
                            yield from self._process_buffer(buf_X, buf_y, buf_meta)
                            buf_X.clear()
                            buf_y.clear()
                            buf_meta.clear()
                            
            except Exception as e:
                print(f"Error processing row group {rg_idx}: {e}")
                continue
        
        # Process remaining buffer
        if buf_X:
            yield from self._process_buffer(buf_X, buf_y, buf_meta)
    
    def _process_buffer(self, buf_X, buf_y, buf_meta):
        """Process accumulated buffer into batches"""
        Xb = np.array(buf_X)
        yb = np.array(buf_y, dtype=np.float32)
        meta = np.array(buf_meta, dtype=object)
        
        # Create batches
        n_samples = len(yb)
        n_batches = (n_samples + self.batch_size - 1) // self.batch_size
        
        for i in range(n_batches):
            start_idx = i * self.batch_size
            end_idx = min(start_idx + self.batch_size, n_samples)
            
            batch_X = Xb[start_idx:end_idx]
            batch_y = yb[start_idx:end_idx]
            batch_meta = meta[start_idx:end_idx]
            
            # Apply oversampling if requested
            if self.pos_ratio and len(batch_y) >= 10:  # Only if batch is large enough
                pos_indices = np.where(batch_y == 1)[0]
                neg_indices = np.where(batch_y == 0)[0]
                
                if len(pos_indices) > 0 and len(neg_indices) > 0:
                    n_pos = int(len(batch_y) * self.pos_ratio)
                    n_neg = len(batch_y) - n_pos
                    
                    sel_pos = np.random.choice(pos_indices, min(n_pos, len(pos_indices)), replace=n_pos > len(pos_indices))
                    sel_neg = np.random.choice(neg_indices, min(n_neg, len(neg_indices)), replace=n_neg > len(neg_indices))
                    
                    selected = np.concatenate([sel_pos, sel_neg])
                    np.random.shuffle(selected)
                    
                    batch_X = batch_X[selected]
                    batch_y = batch_y[selected]
                    batch_meta = batch_meta[selected]
            
            # Scale and clip
            B = batch_X.shape[0]
            flat = batch_X.reshape(-1, batch_X.shape[-1])
            flat = self.feature_scaler.transform(flat)
            flat = np.clip(flat, -5.0, 5.0)
            batch_X = flat.reshape(B, self.window, -1)
            
            # Convert to tensors
            xb = torch.from_numpy(batch_X)
            yb_tensor = torch.from_numpy(batch_y)
            
            yield xb, yb_tensor, batch_meta

# ────────────────────────────────────────────────────────────
# 6) Model definition (unchanged)
# ────────────────────────────────────────────────────────────
class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, d_model, ticker_vocab=None):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_features, d_model))
        self.bias   = nn.Parameter(torch.zeros(num_features, d_model))
        nn.init.trunc_normal_(self.weight, std=0.02)
        self.col_emb = nn.Parameter(torch.randn(num_features, d_model) * 0.02)
        if ticker_vocab is not None:
            self.ticker_emb = nn.Embedding(ticker_vocab, d_model)
            self.with_ticker = True
        else:
            self.with_ticker = False

    def forward(self, x, tick_id=None):
        tok = x.unsqueeze(-1) * self.weight + self.bias + self.col_emb
        if self.with_ticker:
            tok = torch.cat(
                [self.ticker_emb(tick_id).unsqueeze(1), tok],
                dim=1
            )
        return tok

class FTTransformer(nn.Module):
    def __init__(self, num_features, d_model=128, n_heads=8,
                 depth=6, mlp_ratio=2.0, dropout=0.2, ticker_vocab=None):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_model, ticker_vocab)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=int(d_model*mlp_ratio),
            dropout=dropout, batch_first=True,
            activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, depth)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x, tick_id=None):
        tok = self.tokenizer(x, tick_id)
        h   = self.encoder(tok)
        cls = h[:, 0] if self.tokenizer.with_ticker else h.mean(dim=1)
        return self.cls_head(cls).squeeze(1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction
    def forward(self, logits, targets):
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p  = torch.sigmoid(logits)
        p_t= targets*p + (1-targets)*(1-p)
        loss = self.alpha * (1-p_t).pow(self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

Using cuda
Dates→ train 12784  |  test 3196
37 numeric feature columns
11526 unique tickers
Cutoff map for 15980 dates (train + test)


## Train

In [ ]:
# ────────────────────────────────────────────────────────────
# 7) Training setup with optimized DataLoader
# ────────────────────────────────────────────────────────────
train_ds = ParquetSequenceDataset(
    date_subset=train_dates,
    feature_cols=feature_cols,
    return_col=return_col,
    cutoff_map=cutoff_map,
    feature_scaler=feature_scaler,
    window=10,
    pos_ratio=0.30,
    batch_size=batch_size
)

# Calculate steps per epoch
total_rows = sum(pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups))
train_steps = int((total_rows * 0.8) // batch_size)

# Optimized worker count
num_workers = min(4, max(1, os.cpu_count() - 2))
print(f"Using {num_workers} workers")

# Configure DataLoader with optimizations
train_loader = DataLoader(
    train_ds, 
    batch_size=None,  # Dataset handles batching
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
    pin_memory=True,
    prefetch_factor=2 if num_workers > 0 else 2,
    drop_last=False
)

# ────────────────────────────────────────────────────────────
# 8) Model and optimizer setup
# ────────────────────────────────────────────────────────────
model = FTTransformer(
    num_features=len(feature_cols)*10,
    d_model=128, n_heads=8, depth=6,
    dropout=0.2, ticker_vocab=ticker_vocab
).to(device)

# Optimize CUDA
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

criterion = FocalLoss(alpha=0.25, gamma=2.0).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    steps_per_epoch=train_steps, epochs=n_epochs, pct_start=0.2
)
scaler = GradScaler()

# ────────────────────────────────────────────────────────────
# 9) Training loop with robust notebook progress tracking
# ────────────────────────────────────────────────────────────
for epoch in range(1, n_epochs+1):
    model.train()
    running_loss = 0.0
    step_count = 0
    
    # Initialize progress bar
    pbar = tqdm(total=train_steps, desc=f"Epoch {epoch}/{n_epochs}", mininterval=0.5)
    
    try:
        for i, (xb, yb, meta) in enumerate(train_loader):
            if step_count >= train_steps:  # Ensure we don't exceed steps
                break
                
            yb = yb.to(device, non_blocking=True)
            tick_ids = torch.tensor(
                [ticker_id_map[t] for t in meta[:,1]],
                dtype=torch.long, device=device
            )
            
            optimizer.zero_grad(set_to_none=True)
            
            with autocast(device_type=device.type, dtype=torch.float16):
                xb_flat = xb.to(device, non_blocking=True).flatten(1)
                logits = model(xb_flat, tick_ids)
                loss = criterion(logits, yb)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            # Update metrics
            running_loss += loss.item()
            step_count += 1
            
            # Update progress bar
            pbar.update(1)
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'avg_loss': f"{(running_loss/step_count):.4f}"
            })
            
            # Periodic console output
            if step_count % 100 == 0:
                avg_loss = running_loss / step_count
                print(f"Step {step_count}/{train_steps} | Avg Loss: {avg_loss:.4f}")
                
    except Exception as e:
        print(f"\nTraining error at epoch {epoch}, step {step_count}: {str(e)}")
        raise e
    finally:
        pbar.close()
    
    avg_loss = running_loss / max(step_count, 1)
    print(f"→ Epoch {epoch} mean loss: {avg_loss:.4f}")

# ────────────────────────────────────────────────────────────
# 10) Save model with error handling
# ────────────────────────────────────────────────────────────
model.eval()
os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/TopSelector_state_dict.pt")

# Create traced model
with torch.no_grad():
    traced = torch.jit.trace(
        model,
        (torch.randn(1, len(feature_cols)*10, device=device),
            torch.tensor([0], dtype=torch.long, device=device)),
        check_trace=False
    )
    traced.save("models/TopSelector.pt")
print("✓ Saved both state dict and traced model")
    

print("Training completed!")

Using 4 workers


C:\Users\taren\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Epoch 1/15:   0%|          | 0/333205 [00:00<?, ?it/s]

## Test

In [ ]:
# mode = 
test_ds = ParquetSequenceDataset(
    date_subset=test_dates,
    feature_cols=feature_cols,
    return_col=return_col,
    cutoff_map=cutoff_map,
    feature_scaler=feature_scaler,
    window=10,
    pos_ratio=None,
    batch_size=batch_size
)
test_loader  = DataLoader(test_ds , batch_size=None, num_workers=0)
# ────────────────────────────────────────────────────────────
# 10) Inference & metrics (corrected)
# ────────────────────────────────────────────────────────────
y_true, y_prob, rows = [], [], []

model.eval()
with torch.no_grad():
    for xb, yb, meta in tqdm(test_loader, desc="Testing", ncols=90):
        yb = yb.to(device)
        tick_ids = torch.tensor(
            [ticker_id_map[t] for t in meta[:,1]],
            dtype=torch.long, device=device
        )
        # xb: [B, window, F] → flatten to [B, F*window]
        xb_flat = xb.view(xb.size(0), -1).to(device)       # now shape [B, F*10]
        logits  = model(xb_flat, tick_ids)      # feed into your existing FTTransformer
        probs  = torch.sigmoid(logits).cpu().numpy()
        y_true.extend(yb.cpu().numpy())
        y_prob.extend(probs)
        rows.extend(zip(meta[:,0], meta[:,1], probs, yb.cpu().numpy()))

test_df = pd.DataFrame(rows, columns=["date", "ticker", "prob", "label"])

# ensure label is int, not float
test_df["label"] = test_df["label"].astype(int)

# ─── precision@5 % per day ───────────────────────────────────
test_df["rank_in_day"] = test_df.groupby("date")["prob"]\
                                .rank(method="first", ascending=False)

def flag_top_frac(group, frac=0.05):
    k   = max(1, int(len(group) * frac))
    idx = group.nsmallest(k, "rank_in_day").index
    s   = pd.Series(0, index=group.index, dtype=int)
    s.loc[idx] = 1
    return s

# clean apply with group_keys=False and explicit frac
test_df["pred_top5"] = (
    test_df.groupby("date", group_keys=False)
           .apply(flag_top_frac, frac=0.05, include_groups=False)   # ← add kwarg
)

# daily precision
hits_per_day  = (
    test_df[test_df["pred_top5"] == 1]
           .groupby("date")["label"]
           .mean()
)
avg_precision = hits_per_day.mean()

# overall recall — boolean mask on ints
true_positives = ((test_df["pred_top5"] == 1) & (test_df["label"] == 1)).sum()
overall_recall = true_positives / test_df["label"].sum()

# ─── full metrics ────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    roc_auc_score, classification_report
)

y_true_arr = np.array(y_true, dtype=int)
y_prob_arr = np.array(y_prob)
y_pred_arr = (y_prob_arr >= 0.5).astype(int)

print("\n──────────  METRICS  ──────────")
print("Overall accuracy       :", accuracy_score(y_true_arr, y_pred_arr))
print("Balanced accuracy      :", balanced_accuracy_score(y_true_arr, y_pred_arr))
print("AUROC                  :", roc_auc_score(y_true_arr, y_prob_arr))
print(classification_report(y_true_arr, y_pred_arr, digits=3))
print(f"Precision@5 % (daily)  : {avg_precision:.3f}")
print(f"Recall (overall)       : {overall_recall:.3f}")

# ─── random baseline ────────────────────────────────────────
rand_hits = (
    test_df.groupby("date")["label"]
           .apply(lambda labs: np.random.choice(
               labs.values,
               size=max(1, int(0.05 * len(labs))),
               replace=False
           ).mean())
).mean()
print(f"Random precision@5 %   : {rand_hits:.3f}")

Testing: 1206it [03:38,  5.52it/s]



──────────  METRICS  ──────────
Overall accuracy       : 0.8108168088231155
Balanced accuracy      : 0.7078180919705563
AUROC                  : 0.8069260816529579
              precision    recall  f1-score   support

           0      0.973     0.823     0.892   4678560
           1      0.157     0.593     0.248    260013

    accuracy                          0.811   4938573
   macro avg      0.565     0.708     0.570   4938573
weighted avg      0.930     0.811     0.858   4938573

Precision@5 % (daily)  : 0.211
Recall (overall)       : 0.200
Random precision@5 %   : 0.052


## Scores

21% Accuracy
